# FINPLE universe v2 delta onboarding

Self-contained operator-only Colab for PR #344. Every risky cell is fail-closed
until its exact confirmation token is set. Do not run provider, Drive-write, or
candidate-package cells during code review.


In [ ]:
# 1. Settings and confirmation gates
from pathlib import Path
from datetime import date
import hashlib
import json
import os
import re
import subprocess
import sys

DRIVE_ROOT = Path("/content/drive/MyDrive/FINPLE/monthly-metrics")
SOURCE_SNAPSHOT_DATE = "2026-07-22"
TARGET_UNIVERSE_VERSION = "finple-universe-v2-2026-07-24"
ATTEMPT_ID = ""  # Set a fresh whitespace-free operator attempt ID.

REPOSITORY_URL = "https://github.com/vip930sw/FINPLE.git"
REPOSITORY_ROOT = Path("/content/FINPLE")
OPERATOR_BRANCH = "codex/step114-2zb-dynamic-universe-megacap-income"
EXPECTED_OPERATOR_HEAD = ""  # Paste the exact reviewed 40-character PR head.

CONFIRMATIONS = {
    "public_clone": "",
    "branch_checkout": "",
    "dependency_install": "",
    "drive_mount": "",
    "canonical_check": "",
    "provider_collection": "",
    "resume_integrity": "",
    "us_only_merge": "",
    "candidate_preparation": "",
    "candidate_package": "",
    "temporary_cleanup": "",
    "artifact_report": "",
}
REQUIRED_CONFIRMATIONS = {
    "public_clone": "CLONE_PUBLIC_FINPLE",
    "branch_checkout": "CHECKOUT_EXACT_PR_HEAD",
    "dependency_install": "INSTALL_OPERATOR_DEPENDENCIES",
    "drive_mount": "MOUNT_FINPLE_DRIVE",
    "canonical_check": "CHECK_CANONICAL_V2",
    "provider_collection": "RUN_29_US_PROVIDER_COLLECTION",
    "resume_integrity": "VERIFY_DELTA_ARTIFACT_INTEGRITY",
    "us_only_merge": "MERGE_US_DELTA_LOCALLY",
    "candidate_preparation": "PREPARE_REVIEW_ONLY_CANDIDATE_INPUTS",
    "candidate_package": "RUN_REVIEW_ONLY_CANDIDATE_PACKAGE",
    "temporary_cleanup": "DELETE_LOCAL_MERGED_US_RAW",
    "artifact_report": "REPORT_GENERATED_ARTIFACTS",
}

def require_confirmation(name):
    expected = REQUIRED_CONFIRMATIONS[name]
    if CONFIRMATIONS.get(name) != expected:
        raise RuntimeError(f"Fail closed: set CONFIRMATIONS[{name!r}] to {expected!r}")
    if not ATTEMPT_ID or any(character.isspace() for character in ATTEMPT_ID):
        raise RuntimeError("Fail closed: set a fresh whitespace-free ATTEMPT_ID")

def run_visible(command, label, cwd):
    print({"label": label, "command": [str(part) for part in command], "cwd": str(cwd)}, flush=True)
    result = subprocess.run(
        [str(part) for part in command],
        cwd=Path(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )
    print(result.stdout, end="" if result.stdout.endswith("\n") or not result.stdout else "\n")
    print(result.stderr, end="" if result.stderr.endswith("\n") or not result.stderr else "\n")
    if result.returncode != 0:
        raise RuntimeError(f"{label} failed with exit status {result.returncode}")
    return result


In [ ]:
# 2. Public GitHub clone
require_confirmation("public_clone")
if REPOSITORY_ROOT.exists():
    raise RuntimeError(f"Fail closed: clone target already exists: {REPOSITORY_ROOT}")
run_visible(
    ["git", "clone", "--filter=blob:none", "--no-checkout", REPOSITORY_URL, REPOSITORY_ROOT],
    "public GitHub clone",
    Path("/content"),
)


In [ ]:
# 3. Exact branch fetch and detached checkout
require_confirmation("branch_checkout")
run_visible(
    ["git", "fetch", "--depth", "1", "origin", OPERATOR_BRANCH],
    "fetch exact operator branch",
    REPOSITORY_ROOT,
)
run_visible(
    ["git", "checkout", "--detach", "FETCH_HEAD"],
    "checkout fetched operator head",
    REPOSITORY_ROOT,
)


In [ ]:
# 4. Assert repository HEAD equals the operator-reviewed head
if not re.fullmatch(r"[0-9a-f]{40}", EXPECTED_OPERATOR_HEAD):
    raise RuntimeError("Fail closed: EXPECTED_OPERATOR_HEAD must be an exact 40-character SHA")
ACTUAL_OPERATOR_HEAD = run_visible(
    ["git", "rev-parse", "HEAD"],
    "read repository HEAD",
    REPOSITORY_ROOT,
).stdout.strip()
if ACTUAL_OPERATOR_HEAD != EXPECTED_OPERATOR_HEAD:
    raise RuntimeError(
        f"Fail closed: repository HEAD mismatch; expected {EXPECTED_OPERATOR_HEAD}, got {ACTUAL_OPERATOR_HEAD}"
    )
os.chdir(REPOSITORY_ROOT)
print({"operatorBranch": OPERATOR_BRANCH, "operatorHead": ACTUAL_OPERATOR_HEAD})


In [ ]:
# 5. Install only the operator dependencies
require_confirmation("dependency_install")
run_visible(
    [sys.executable, "-m", "pip", "install", "yfinance", "pandas"],
    "install operator dependencies",
    REPOSITORY_ROOT,
)


In [ ]:
# 6. Mount Google Drive
require_confirmation("drive_mount")
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# 7. Existing canonical three-file source preflight
from scripts.finple_universe_delta_operator import preflight_combined_sources

SOURCE_COMBINED_ROOT = DRIVE_ROOT / SOURCE_SNAPSHOT_DATE / "combined"
SOURCE_INVENTORY = preflight_combined_sources(SOURCE_COMBINED_ROOT)
for role, record in SOURCE_INVENTORY.items():
    print({
        "role": role,
        "path": str(record["path"]),
        "sizeBytes": record["sizeBytes"],
        "sha256": record["sha256"],
    })


In [ ]:
# 8. Phase A deterministic canonical check
require_confirmation("canonical_check")
CANONICAL_V2 = REPOSITORY_ROOT / "src/data/tickers/finple_app_candidates_v2.csv"
RECONCILIATION = REPOSITORY_ROOT / "src/data/tickers/finple_universe_v2_reconciliation.json"
run_visible(
    [
        sys.executable, "-m", "scripts.finple_universe_v2",
        "--source", "src/data/tickers/finple_app_candidates_6000_balanced_v1.csv",
        "--output", "src/data/tickers/finple_app_candidates_v2.csv",
        "--manifest", "src/data/tickers/finple_universe_v2_manifest.json",
        "--reconciliation", "src/data/tickers/finple_universe_v2_reconciliation.json",
        "--check",
    ],
    "canonical v2 deterministic check",
    REPOSITORY_ROOT,
)


In [ ]:
# 9. Phase B collect only the 29 new US identities
require_confirmation("provider_collection")
COLLECT_COMMAND = [
    sys.executable, "-m", "scripts.collect_finple_universe_delta",
    "--canonical", str(CANONICAL_V2),
    "--reconciliation", str(RECONCILIATION),
    "--drive-root", str(DRIVE_ROOT),
    "--target-version", TARGET_UNIVERSE_VERSION,
]
run_visible(COLLECT_COMMAND, "collect universe delta", REPOSITORY_ROOT)


In [ ]:
# 10. Delta artifact integrity and resume preflight
require_confirmation("resume_integrity")
run_visible(
    [*COLLECT_COMMAND, "--resume"],
    "verify existing delta artifact integrity",
    REPOSITORY_ROOT,
)
DELTA_ROOT = DRIVE_ROOT / "universe-deltas" / TARGET_UNIVERSE_VERSION
DELTA_RAW = DELTA_ROOT / "us-new-assets-raw-daily.csv"
BENCHMARK_ADDITIONS = DELTA_ROOT / "benchmark-additions.csv"


In [ ]:
# 11. Phase C US-only streaming merge into local /content
require_confirmation("us_only_merge")
MERGED_US_RAW = Path("/content/finple-universe-v2-us-merged-raw.csv")
MERGE_RECONCILIATION = Path(f"/content/finple-universe-v2-us-merge-{ATTEMPT_ID}.json")
if MERGED_US_RAW.exists() or MERGE_RECONCILIATION.exists():
    raise RuntimeError("Fail closed: local merge output already exists")
run_visible(
    [
        sys.executable, "-m", "scripts.merge_finple_universe_delta",
        "--source", str(SOURCE_INVENTORY["usRaw"]["path"]),
        "--delta", str(DELTA_RAW),
        "--output", str(MERGED_US_RAW),
        "--reconciliation", str(MERGE_RECONCILIATION),
    ],
    "US-only streaming merge",
    REPOSITORY_ROOT,
)


In [ ]:
# 12. Bind existing KR raw and overlay as read-only inputs
from scripts.finple_universe_delta_operator import assert_sources_unchanged

KR_RAW_READ_ONLY = Path(SOURCE_INVENTORY["krRaw"]["path"])
KR_OVERLAY_READ_ONLY = Path(SOURCE_INVENTORY["krOverlay"]["path"])
assert_sources_unchanged(SOURCE_INVENTORY)
print({
    "krRawReadOnly": str(KR_RAW_READ_ONLY),
    "krOverlayReadOnly": str(KR_OVERLAY_READ_ONLY),
})


In [ ]:
# 13. Prepare the exact five-input One-Click candidate contract
require_confirmation("candidate_preparation")
from scripts.finple_universe_delta_operator import build_candidate_prepare_command

CANDIDATE_INPUT_DIR = Path(f"/content/finple-universe-v2-candidate-inputs/{ATTEMPT_ID}")
CANDIDATE_INPUT_REPORT = Path(
    f"/content/finple-universe-v2-candidate-input-reconciliation-{ATTEMPT_ID}.json"
)
PREPARE_COMMAND = build_candidate_prepare_command(
    repository_root=REPOSITORY_ROOT,
    canonical_v2=CANONICAL_V2,
    merged_us_raw=MERGED_US_RAW,
    kr_raw=KR_RAW_READ_ONLY,
    kr_overlay=KR_OVERLAY_READ_ONLY,
    benchmark_additions=BENCHMARK_ADDITIONS,
    output_dir=CANDIDATE_INPUT_DIR,
    report=CANDIDATE_INPUT_REPORT,
    metric_base_date=SOURCE_SNAPSHOT_DATE,
    attempt_id=ATTEMPT_ID,
    python_executable=sys.executable,
)
run_visible(PREPARE_COMMAND, "prepare candidate inputs", REPOSITORY_ROOT)
assert_sources_unchanged(SOURCE_INVENTORY)


In [ ]:
# 14. Run the review-only One-Click candidate package
require_confirmation("candidate_package")
from scripts.metrics_pipeline import run_finple_production_candidate_package
from scripts.metrics_pipeline.config import PARTIAL_MONTH_POLICY

CANDIDATE_OUTPUT_DIR = Path(f"/content/finple-universe-v2-one-click/{ATTEMPT_ID}")
if CANDIDATE_OUTPUT_DIR.exists():
    raise RuntimeError(f"Fail closed: candidate output already exists: {CANDIDATE_OUTPUT_DIR}")
CANDIDATE_CONFIG = {
    "metric_base_date": SOURCE_SNAPSHOT_DATE,
    "market_scope": ["US", "KR"],
    "output_version": f"{TARGET_UNIVERSE_VERSION}-{ATTEMPT_ID}",
    "input_mode": "manual_upload_candidate",
    "input_dir": str(CANDIDATE_INPUT_DIR),
    "output_dir": str(CANDIDATE_OUTPUT_DIR),
    "validation_date": date.today().isoformat(),
    "internal_preview_review_only": True,
    "partial_month_policy": PARTIAL_MONTH_POLICY,
}
RESULT = run_finple_production_candidate_package(CANDIDATE_CONFIG)
if RESULT.get("productionPublishReady") or RESULT.get("appExportApproved"):
    raise RuntimeError("Fail closed: candidate package opened a production/publication gate")
assert_sources_unchanged(SOURCE_INVENTORY)
print({
    "candidatePackageReady": RESULT.get("candidatePackageReady", False),
    "productionPublishReady": RESULT.get("productionPublishReady", False),
    "appExportApproved": RESULT.get("appExportApproved", False),
})


In [ ]:
# 15. Delete only the local temporary merged US raw
require_confirmation("temporary_cleanup")
from scripts.finple_universe_delta_operator import cleanup_temporary_merged_us

if not RESULT.get("candidatePackageReady", False):
    raise RuntimeError("Fail closed: preserve merged US raw until candidate package succeeds")
assert_sources_unchanged(SOURCE_INVENTORY)
cleanup_temporary_merged_us(MERGED_US_RAW, local_root=Path("/content"))
assert_sources_unchanged(SOURCE_INVENTORY)
print({"temporaryMergedUsDeleted": not MERGED_US_RAW.exists()})


In [ ]:
# 16. Report generated paths, sizes, and SHA-256
require_confirmation("artifact_report")
from scripts.finple_universe_delta_operator import sha256_file

GENERATED_PATHS = [
    MERGE_RECONCILIATION,
    CANDIDATE_INPUT_REPORT,
    *sorted(path for path in CANDIDATE_INPUT_DIR.rglob("*") if path.is_file()),
    *sorted(path for path in CANDIDATE_OUTPUT_DIR.rglob("*") if path.is_file()),
]
for path in GENERATED_PATHS:
    print({
        "path": str(path),
        "sizeBytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })
assert_sources_unchanged(SOURCE_INVENTORY)
print({
    "temporaryMergedUsDeleted": not MERGED_US_RAW.exists(),
    "krRawUnchanged": True,
    "krOverlayUnchanged": True,
    "productionPublishReady": False,
    "appExportApproved": False,
})
